In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w9" / "day01"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

SAMPLES = ROOT / "sandbox" / "w5" / "day01" / "samples"
FORMATS = SAMPLES / "_formats"

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


In [2]:
from app.agent.question_fix import FIX_WORDS, fix_question

print(fix_question({"question": "출장 가서 호텔비는 얼마까지 되나요?"}))

{'query': '출장 가서 숙박비는 얼마까지 되나요?', 'fixes': 1}


In [3]:
from app.agent.question_fix import fix_question
from app.agent.state import LoopState
from app.agent.graph import build_graph
from app.agent.patterns import route_after_search
from langgraph.graph import END, START, StateGraph

RULE_WORDS = ("숙박비", "식비", "교통비") 
QUESTIONS = ["부산 출장 숙박비 한도가 얼마인가요?", "출장 가서 호텔비는 얼마까지 되나요?", "점심 메뉴"]


def fake_search(state):
    q = state.get("query") or state["question"]
    return {"passed": any(w in q for w in RULE_WORDS), "trace": ["search"]} 

def fake_end(name):
    def step(state):
        return {"trace": [name]}
    return step

def traced(name, step):

    def wrapped(state):
        return {**step(state), 'trace': [name]}
    return wrapped

def pick_no_limit(state):
    return 'found' if state.get('passed') else 'fix'

g = StateGraph(LoopState)
g.add_node('search', fake_search)
g.add_node('fix', traced('fix', fix_question))
g.add_node('found', fake_end('found'))
g.add_edge(START, 'search')
g.add_conditional_edges('search', pick_no_limit, {'found': 'found', 'fix': 'fix'})
g.add_edge('found', END)
g.add_edge('fix', 'search')
loop = g.compile()
out = loop.invoke({'question': QUESTIONS[1], 'user': {}})
print('trace :', out['trace'])
print('query :', out['query'])
print('fixes :', out['fixes'])


trace : ['search', 'fix', 'search', 'found']
query : 출장 가서 숙박비는 얼마까지 되나요?
fixes : 1


In [4]:
from alembic import command
from alembic.config import Config
from sqlalchemy import inspect
from app.db.session import get_engine
cfg_a = Config('backend/alembic.ini')
VERS = ROOT / 'backend' / 'app' / 'db' / 'migrations' / 'versions'
if not list(VERS.glob('*_add_approval_tables.py')):
    command.revision(cfg_a, message='add approval tables', autogenerate=True)
rev_text = next(VERS.glob('*_add_approval_tables.py')).read_text(encoding='utf-8')
up_body = rev_text.split('def upgrade()')[1].split('def downgrade()')[0]
drops = sum(('drop_' in line for line in up_body.splitlines()))
if drops:
    raise RuntimeError('upgrade() 에 drop_ — 3절 include_name 확인')
command.upgrade(cfg_a, 'head')
print('DB 표 :', len(inspect(get_engine()).get_table_names()))
print(f"마이그레이션 : {len(list(VERS.glob('*.py')))}건")
print(f'upgrade() 안의 drop_ : {drops}줄')


INFO  [alembic.runtime.migration] Context impl PostgresqlImpl.
INFO  [alembic.runtime.migration] Will assume transactional DDL.


DB 표 : 19
마이그레이션 : 10건
upgrade() 안의 drop_ : 0줄


In [5]:
from langgraph.checkpoint.postgres import PostgresSaver  
from sqlalchemy import inspect
from app.core.config import get_settings
from app.db.session import get_engine

CKPT = {"checkpoints", "checkpoint_blobs", "checkpoint_writes", "checkpoint_migrations"}
before = set(inspect(get_engine()).get_table_names())

PG_URL = get_settings().database_url.replace("postgresql+psycopg://", "postgresql://", 1)
with PostgresSaver.from_conn_string(PG_URL) as saver:      
    saver.setup()                                        

after = set(inspect(get_engine()).get_table_names())
print(f"DB 표         : {len(before)} → {len(after)}")
print(f"체크포인트 표 : {len(CKPT & after)}/4  {sorted(CKPT & after)}")
print("새로 만든 표  :", sorted(after - before) or "없음 — 이미 있었습니다")

DB 표         : 19 → 19
체크포인트 표 : 4/4  ['checkpoint_blobs', 'checkpoint_migrations', 'checkpoint_writes', 'checkpoints']
새로 만든 표  : 없음 — 이미 있었습니다


In [6]:
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command
import importlib

import app.agent.graph as graph_module
importlib.reload(graph_module)

from app.agent.graph import build_approval_graph


graph = build_approval_graph(lambda s: {"text": "부산 출장 승인 요청"}, lambda s: {"result": {"ok": True}}, checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "레포-그래프"}}
print("멈췄나 :", "__interrupt__" in graph.invoke({"text": ""}, cfg))
print("승인 뒤 결과 :", graph.invoke(Command(resume={"approved": True, "note": ""}), cfg)["result"])
try:
    build_approval_graph(lambda s: s, lambda s: s, checkpointer=None)
except ValueError as e:
    print("저장소 없이 : ValueError ·", str(e).split(" — ")[0]) 

멈췄나 : True
승인 뒤 결과 : {'ok': True}
저장소 없이 : ValueError · 승인 그래프는 저장소가 있어야 멈췄다 이어 간다. checkpointer를 넘기세요.


In [7]:
import uuid
from langgraph.checkpoint.postgres import PostgresSaver
from app.core.config import get_settings
import app.services.chat_service as chat_service
import app.services.approval_service as approval_service

PG_URL = get_settings().database_url.replace("postgresql+psycopg://", "postgresql://", 1)  

with PostgresSaver.from_conn_string(PG_URL) as saver:
    first = chat_service.run_approval_graph(text="부산 출장 신청서 승인 요청 — 10/19 오전", thread_id=uuid.uuid4().hex, requester_id=1, checkpointer=saver)
    
print("요청 :", first["req_id"], "·", first["status"])
print("감사 로그 :", approval_service.events(first["req_id"]))

요청 : REQ-0159 · 대기중
감사 로그 : ['Slack 전송 요청 → 승인 대기']


In [8]:
from app.agent.graph import build_supervisor_graph
from app.agent.supervisor import MAX_TURNS, pick_worker

# 가짜 검색 도구 
def fake_search(state):
    return {'sources': [{'doc_id': '가짜'}], 'passed': True, 'trace': ['search']}

# 가짜 초안 도구 
def fake_draft(state):
    return {'draft': {'body': '가짜 초안'}, 'trace': ['draft']}

sv = build_supervisor_graph({'search': fake_search, 'draft': fake_draft}, pick_worker)

for q in ('출장 숙박비 상한은 얼마인가요?', '부산 출장 신청서를 써 줘'):
    out = sv.invoke({'question': q, 'user': {}}, config={'recursion_limit': 2 * MAX_TURNS + 2})
    print(q, '→', out['trace'], '· turns', out['turns'])


출장 숙박비 상한은 얼마인가요? → ['search'] · turns 2
부산 출장 신청서를 써 줘 → ['search', 'draft'] · turns 3


In [9]:
import uuid
from langgraph.checkpoint.postgres import PostgresSaver
from app.core.config import get_settings
from app.services import approval_service, chat_service
PG_URL = get_settings().database_url.replace('postgresql+psycopg://', 'postgresql://', 1)
with PostgresSaver.from_conn_string(PG_URL) as saver:
    req = chat_service.run_approval_graph(text='부산 출장 신청서 승인 요청 — 반려 해 보기', thread_id=uuid.uuid4().hex, requester_id=1, checkpointer=saver)
    out = approval_service.reject(req_id=req['req_id'], approver_id=2, note='출장지 확인', checkpointer=saver)
print('반려 뒤 :', out['status'])
print('감사 로그 :', approval_service.events(req['req_id']))


반려 뒤 : 대기중
감사 로그 : ['Slack 전송 요청 → 승인 대기', '반려', '재작성']


In [10]:
import importlib
import uuid
from langgraph.checkpoint.postgres import PostgresSaver
from app.core.config import get_settings
from app.services import approval_service, chat_service
importlib.reload(approval_service)

PG_URL = get_settings().database_url.replace('postgresql+psycopg://', 'postgresql://', 1)

with PostgresSaver.from_conn_string(PG_URL) as saver:
    req = chat_service.run_approval_graph(text='부산 출장 신청서 승인 요청 10/27', thread_id=uuid.uuid4().hex, requester_id=1, checkpointer=saver)
    r1 = approval_service.reject(req_id=req['req_id'], approver_id=2, note='금액 확인', checkpointer=saver)
    print('반려 뒤 :', r1['status'], '·', r1['text'])
    r2 = approval_service.approve(req_id=req['req_id'], approver_id=2, checkpointer=saver)
    print('승인 뒤 :', r2['status'])
print('감사 로그 :', approval_service.events(req['req_id']))


반려 뒤 : 대기중 · 부산 출장 신청서 승인 요청 10/27 (사유 반영: 금액 확인)
14:39:56 INFO     app.integrations.slack : Slack 전송 완료: #infra-team2
승인 뒤 : 승인·실행
감사 로그 : ['Slack 전송 요청 → 승인 대기', '반려', '재작성', '승인', 'Slack 전송 완료']


In [11]:
GOLDEN = '이번 프로젝트 출장비 기준을 찾아서 부산 출장 신청서를 작성하고, 팀장 승인 요청 메시지도 만들어줘.'

sv = chat_service.run_supervisor_graph(question=GOLDEN)

text = f"{sv['draft']['doc_type']} 승인 요청 ({sv['run_id']})" if 'draft' in sv else GOLDEN
PG_URL = get_settings().database_url.replace('postgresql+psycopg://', 'postgresql://', 1)
with PostgresSaver.from_conn_string(PG_URL) as saver:
    req = chat_service.run_approval_graph(text=text, thread_id=uuid.uuid4().hex, requester_id=1, checkpointer=saver, run_id=sv.get('run_id'))
    done = approval_service.approve(req_id=req['req_id'], approver_id=2, checkpointer=saver)
print(f"supervisor : 초안 {('있음' if 'draft' in sv else '없음')} · 고른 횟수 {sv['turns']}")
print('요청 :', req['req_id'], '·', req['status'])
print('승인 :', done['status'])
print(f"감사 로그 : {len(approval_service.events(req['req_id']))}줄")


14:40:01 INFO     sentence_transformers.base.model : No device provided, using mps
14:40:01 INFO     sentence_transformers.base.model : Loading SentenceTransformer model from ../models/bge-m3.


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

14:40:03 INFO     app.integrations.local_embed : 로컬 임베딩 모델 준비: ../models/bge-m3 / 1024차원


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

14:40:03 INFO     app.rag.retriever : 정책으로 제외: 국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전
14:40:03 INFO     sentence_transformers.base.model : No device provided, using mps
14:40:03 INFO     sentence_transformers.base.model : No modules.json found for ../models/bge-reranker-v2-m3, initializing a new CrossEncoder model.


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

14:40:05 INFO     app.integrations.local_rerank : 로컬 리랭커 준비 : ../models/bge-reranker-v2-m3, 최대 512토큰
14:40:05 INFO     app.integrations.slack : Slack 전송 완료: #infra-team2
supervisor : 초안 있음 · 고른 횟수 3
요청 : REQ-0162 · 대기중
승인 : 승인·실행
감사 로그 : 3줄


In [12]:
import uuid
from langgraph.checkpoint.postgres import PostgresSaver
from app.core.config import get_settings
from app.services import chat_service

QUESTION = '이번 프로젝트 출장비 기준을 찾아서 부산 출장 신청서를 작성하고, 팀장 승인 요청 메시지도 만들어줘.'

sup = chat_service.run_supervisor_graph(question=QUESTION)
print(f"supervisor : 담당을 고른 횟수 {sup['turns']} · 초안 {('있음' if sup.get('draft') else '없음')}")
PG_URL = get_settings().database_url.replace('postgresql+psycopg://', 'postgresql://', 1)
with PostgresSaver.from_conn_string(PG_URL) as saver:
    req = chat_service.run_approval_graph(text='부산 출장 신청서 승인 요청 — 국내출장 여비 규정 v2.0 기준으로 작성했습니다. 확인 부탁드립니다.', thread_id=uuid.uuid4().hex, requester_id=1, checkpointer=saver, run_id=sup.get('run_id'))
print(f"요청 번호 : {req['req_id']} · {req['status']}")
print(f"보일 문구 : {req['text'][:18]}…")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

14:40:05 INFO     app.rag.retriever : 정책으로 제외: 국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전/국내출장 여비 규정 v1.1 - 만료 버전
supervisor : 담당을 고른 횟수 3 · 초안 있음
요청 번호 : REQ-0163 · 대기중
보일 문구 : 부산 출장 신청서 승인 요청 — …
